Initial Logging

In [1]:
import sys
sys.path.append("..")  # if notebook is in notebooks/, adjust so src/ is importable

from src.utils.config_loader import load_config
config = load_config()
print(config)

{'paths': {'tesseract_cmd': 'C:\\Program Files\\Tesseract-OCR\\tesseract.exe', 'raw_data_dir': 'data/raw', 'processed_data_dir': 'data/processed'}, 'extraction': {'supported_formats': ['.pdf', '.docx', '.txt', '.jpg', '.jpeg', '.png'], 'ocr_dpi': 300, 'min_text_length_for_native_pdf': 20}}


In [2]:
from src.utils.logger import get_logger
logger = get_logger("test")
logger.info("Logger test message")
logger.warning("Logger warning test")

2026-07-27 22:57:46 | INFO     | test | Logger test message
2026-07-27 22:57:46 | WARNING  | test | Logger warning test


In [3]:
import fitz, docx, pytesseract, cv2, pdf2image
from pdf2image import convert_from_path
from PIL import Image
print("All extraction dependencies imported successfully")

All extraction dependencies imported successfully


Files Extraction Tests

In [4]:
from src.extraction.pdf_extractor import extract_text_from_pdf

result = extract_text_from_pdf("../data/samples/Native-PDF.pdf")
print("Metadata:", result["metadata"])
print("\n--- Text preview (first 300 chars) ---")
print(result["text"][:300])

2026-07-27 22:58:08 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/Native-PDF.pdf


Metadata: {'filename': '../data/samples/Native-PDF.pdf', 'page_count': 1, 'char_count': 82}

--- Text preview (first 300 chars) ---
Phuket Rajabhat University 
Thanatorn Auksornphan 
Computer Science 
Senior Year 



In [5]:
from src.extraction.docx_extractor import extract_text_from_docx

result = extract_text_from_docx("../data/samples/Word-Docx.docx")
print("Metadata:", result["metadata"])
print("\n--- Text preview ---")
print(result["text"][:300])

2026-07-27 22:58:08 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples/Word-Docx.docx


Metadata: {'filename': '../data/samples/Word-Docx.docx', 'char_count': 77}

--- Text preview ---
Phuket Rajabhat University
Thanatorn Auksornphan
Computer Science
Senior Year


In [6]:
from src.extraction.txt_extractor import extract_text_from_txt

result = extract_text_from_txt("../data/samples/Text-txt.txt")
print("Metadata:", result["metadata"])
print("\n--- Text preview ---")
print(result["text"][:300])

2026-07-27 22:58:09 | INFO     | src.extraction.txt_extractor | Attempting to extract text from TXT: ../data/samples/Text-txt.txt


Metadata: {'filename': '../data/samples/Text-txt.txt', 'char_count': 78}

--- Text preview ---
Phuket Rajabhat University
Thanatorn Auksornphan
Computer Science
Senior Year



In [7]:
from src.extraction.ocr_extractor import extract_text_from_image

result = extract_text_from_image("../data/samples/PNG-Pic.png")
print("Metadata:", result["metadata"])
print("\n--- Text preview ---")
print(result["text"][:300])

2026-07-27 22:58:10 | INFO     | src.extraction.ocr_extractor | Extracting text via OCR from: ../data/samples/PNG-Pic.png
2026-07-27 22:58:10 | INFO     | src.extraction.ocr_extractor | Preprocessed image: ../data/samples/PNG-Pic.png


Metadata: {'filename': '../data/samples/PNG-Pic.png', 'char_count': 79}

--- Text preview ---
Phuket Rajabhat University
Thanatorn Auksornphan
Computer Science

Senior Year



Full Dispatcher Batch Test

In [8]:
import os
from src.extraction.extractor import extract_text

sample_dir = "../data/samples"
for fname in os.listdir(sample_dir):
    path = os.path.join(sample_dir, fname)
    result = extract_text(path)
    status = "ERROR" if "error" in result["metadata"] else "OK"
    print(f"{fname:30s} -> {status:6s} | char_count={result['metadata'].get('char_count', 0)}")

2026-07-27 22:58:22 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples\Native-PDF.pdf
2026-07-27 22:58:22 | INFO     | src.extraction.ocr_extractor | Extracting text via OCR from: ../data/samples\PNG-Pic.png
2026-07-27 22:58:22 | INFO     | src.extraction.ocr_extractor | Preprocessed image: ../data/samples\PNG-Pic.png


Native-PDF.pdf                 -> OK     | char_count=82


2026-07-27 22:58:28 | INFO     | src.extraction.txt_extractor | Attempting to extract text from TXT: ../data/samples\Text-txt.txt
2026-07-27 22:58:28 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples\Word-Docx.docx


PNG-Pic.png                    -> OK     | char_count=79
Text-txt.txt                   -> OK     | char_count=78
Word-Docx.docx                 -> OK     | char_count=77
